# Creating STAC from Scratch

This notebook walks through a minimal, end-to-end example of building STAC metadata from a single GeoTIFF.

You will:
- create a Catalog and Collection
- generate an Item from raster data
- link objects together with self HREFs
- validate each STAC object
- write item, collection, and catalog JSON files to disk

Use this as a starter pattern for your own datasets by replacing the input filename and metadata fields.

The data for this was created using [a GeoTIFF exporting tool](https://cogniscient.auspatious.com)

![Export image of NDVI](../images/stac_export_geotiff.jpeg)

In [ ]:
from pathlib import Path

from pystac import (
    Catalog,
    Collection,
    Asset,
    Link,
    CatalogType,
    SpatialExtent,
    TemporalExtent,
    Extent,
    Provider,
)

from rio_stac import create_stac_item

from pystac.utils import str_to_datetime
from re import search

In [ ]:
item_dir = "ndvi-demo-collection"
item_tif_name = "cogniscient-2026-08-03-index-nir-red-932px-hatsukaichi.tif"
filename = f"{item_dir}/{item_tif_name}"

# Get the date from the file name, using regular expressions
date_str = search(r"\d{4}-\d{2}-\d{2}", item_tif_name).group()
datetime = str_to_datetime(date_str)

In [ ]:
catalog = Catalog(
    id="demo-catalog",
    description="A demo catalog for the Cloud Native Geospatial EO Workshop",
    title="Demo Catalog",
    catalog_type=CatalogType.RELATIVE_PUBLISHED,
)

catalog

In [ ]:
# We update this below, based on items added to the collection.
# For now, we set the temporal extent to None.
extent = Extent(
    SpatialExtent([[-180, -90, 180, 90]]),  # Cover the whole world
    TemporalExtent(
        [None, None]
    ),
)

collection = Collection(
    id="ndvi-demo-collection",
    description="A demo collection for NDVI data",
    title="NDVI Demo Collection",
    extent=extent,
    catalog_type=CatalogType.RELATIVE_PUBLISHED,
    license="CC-BY-4.0",
    keywords=["demo", "stac", "ndvi"],
    providers=[
        Provider(
            name="Cogniscient",
            roles=["processor"],
            url="https://cogniscient.auspatious.com",
        ),
        Provider(
            name="ESA",
            roles=["producer", "licensor"],
            url="https://sentinel.esa.int/web/sentinel/missions/sentinel-2",
        ),
    ],
)

# Note that you could add an Asset to the collection
# For example, a STAC GeoParquet file, as an alternate
# index for the collection. Here we add an empty XML file
collection.add_asset(
    "meta",
    Asset(
        href="example.xml",
        media_type="application/xml",
        title="Empty XML File",
    ),
)

# We could also add a link to WMS or WMTS, if available.
collection.add_link(
    Link(
        rel="wmts",
        target="https://tileserver.example.com/styles/coastlines/wmts.xml",
        media_type="application/xml",
        title="WMTS Service for this Collection",
        extra_fields={
            "wms:layers": ["ndvi"],
        },
    )
)

collection

In [ ]:
item = create_stac_item(
    filename,
    input_datetime=datetime,
    properties={
        "example": 42.0
    },
    asset_name="ndvi"
)
item

In [ ]:
# Link things together using published HREFs and local save paths
item_json_name = item_tif_name.replace(".tif", ".stac-item.json")
item_json_path = f"{item_dir}/{item_json_name}"
collection_json_path = f"{item_dir}/collection.json"
catalog_json_path = "catalog.json"

base_href = "https://raw.githubusercontent.com/auspatious/cloud-native-geospatial-eo-workshop/main/stac"

catalog.set_self_href(f"{base_href}/catalog.json")
collection.set_self_href(f"{base_href}/{item_dir}/collection.json")

collection.add_item(item)
collection.update_extent_from_items()
catalog.add_child(collection)

item.set_self_href(f"{base_href}/{item_dir}/{item_json_name}")
item.assets["ndvi"].href = item_tif_name

In [ ]:
# Validate things
item.validate()
collection.validate()
catalog.validate()

In [ ]:
# Write to disk
item.save_object(dest_href=item_json_path)
collection.save_object(dest_href=collection_json_path)
catalog.save_object(dest_href=catalog_json_path)

## Finishing up

We can preview the results in a stac-broswer